# Scoring uncertainty & prediction intervals

_Metrics & Evaluation — measuring models, data & statistics_

**A good interval is two things at once: it COVERS the truth as often as promised, and it is as NARROW as possible.**

Every metric on this page is answering one of two questions about an interval, and you must answer both:

       
         
- Does it COVER? If you promise 90%, does the truth really fall inside 90% of the time? That is PICP, and an interval whose coverage matches its promise is called CALIBRATED.
         
- How NARROW is it? Among all intervals that cover correctly, the tighter one is more useful. That is MPIW (and the set size for conformal).
       

---

This notebook builds split-conformal prediction intervals one step at a time, then asks both questions of them — does it cover, and how narrow is it? Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q mapie
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, and the target is a continuous value.

In [ ]:
from sklearn.datasets import load_diabetes

data = load_diabetes(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target summary:")
print(data.target.describe())
data.frame.head()

## Reference implementation — scikit-learn + numpy (notes: MAPIE, properscoring)

We turn an ordinary point regressor into a **90% prediction interval** with split conformal prediction, then score that interval two ways. We build it in four steps: (1) fit a model on a clean three-way split, (2) calibrate the interval half-width, (3) measure coverage (PICP) and width (MPIW), (4) score a quantile with pinball loss.

### Step 1 — Split the data three ways and fit a point model

Split conformal needs a **calibration** set that the fitted model has never seen, separate from both training and test. We carve the data into train / calibration / test, then fit an ordinary `LinearRegression` on the training portion only.

Keeping calibration unseen is what makes the later coverage guarantee valid: the calibration residuals must be exchangeable with the test residuals.

In [ ]:
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

X, y = load_diabetes(return_X_y=True)

# Split off 40% as a temporary pool, then halve it into calibration + test.
# Calibration must be UNSEEN by the fitted model for the guarantee to hold.
X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.4, random_state=0)
X_cal, X_te, y_cal, y_te = train_test_split(X_tmp, y_tmp, test_size=0.5, random_state=0)

model = LinearRegression().fit(X_tr, y_tr)

### Step 2 — Calibrate the conformal half-width

On the calibration set we compute **nonconformity scores** — here just the absolute residuals `|y - yhat|`, which capture the model's typical error magnitude with no distributional assumption.

We then take the score at a finite-sample-corrected rank. The `(n+1)` correction is what makes the achieved coverage provably at least `1 - alpha`. The chosen score `qhat` becomes the interval half-width: every test prediction gets the interval `yhat ± qhat`.

In [ ]:
# ---- SPLIT CONFORMAL: turn a point model into a 90% prediction interval ----
alpha = 0.10                                  # target coverage = 1 - alpha = 0.90

res = np.abs(y_cal - model.predict(X_cal))    # nonconformity scores s_i = |y - yhat|
n = len(res)

k = int(np.ceil((n + 1) * (1 - alpha)))       # finite-sample rank
k = min(k, n)                                  # never index past the last score
qhat = np.sort(res)[k - 1]                     # conformal quantile -> interval half-width

pred = model.predict(X_te)                     # point predictions on the test set
lo = pred - qhat                               # interval lower bound
hi = pred + qhat                               # interval upper bound

### Step 3 — Report coverage (PICP) and width (MPIW) together

An interval is only as good as its coverage. **PICP** (prediction-interval coverage probability) is the fraction of test truths that actually land inside `[lo, hi]` — it should match the promised `1 - alpha`. **MPIW** (mean prediction-interval width) is the average interval width, in the target's units.

Always quote them together: a tight interval that misses is worthless, and you compare width only among intervals that cover at the target rate.

In [ ]:
# ---- PICP (coverage) and MPIW (width): always report TOGETHER ----
def picp(y, lo, hi):
    inside = (y >= lo) & (y <= hi)    # which truths fall inside the interval
    return float(np.mean(inside))     # fraction inside

def mpiw(lo, hi):
    widths = hi - lo                  # width of each interval
    return float(np.mean(widths))     # average interval width

print("target coverage :", 1 - alpha)
print("PICP (achieved) :", round(picp(y_te, lo, hi), 3))
print("MPIW (width)    :", round(mpiw(lo, hi), 1))   # in target units

### Step 4 — Score a single quantile with pinball loss

**Pinball (quantile) loss** scores one quantile prediction at a level `tau`. It penalises being below the truth and being above it by different, asymmetric amounts, so it is minimised exactly when the prediction is the true `tau`-quantile.

Here we ask: how good is the conformal **upper** bound as a 0.95-quantile prediction? Lower pinball is better.

In practice, reach for libraries — see the note in the cell for MAPIE (distribution-free conformal intervals) and properscoring (CRPS for full predictive distributions).

In [ ]:
# ---- PINBALL / QUANTILE LOSS for a quantile prediction q at level tau ----
def pinball(y, q, tau):
    d = y - q                                       # signed error
    over = tau * d                                  # penalty when truth is above q
    under = (tau - 1) * d                            # penalty when truth is below q
    return float(np.mean(np.maximum(over, under)))  # take the larger (asymmetric) penalty

# example: how good is the conformal upper bound as a 0.95-quantile prediction?
print("pinball@0.95    :", round(pinball(y_te, hi, 0.95), 2))

# -----------------------------------------------------------------------------
# IN PRACTICE, reach for libraries:
#   MAPIE -- distribution-free conformal intervals, coverage guaranteed:
#       from mapie.regression import MapieRegressor
#       mapie = MapieRegressor(model, method="base", cv="prefit").fit(X_cal, y_cal)
#       _, intervals = mapie.predict(X_te, alpha=0.10)   # shape (n, 2, 1): lo, hi
#
#   properscoring -- CRPS (Continuous Ranked Probability Score) for full
#   predictive distributions:
#       import properscoring as ps
#       crps = ps.crps_ensemble(y_te, samples).mean()    # samples: (n, n_draws)
#       # or, for a Gaussian forecast mean mu, std sigma:
#       crps = ps.crps_gaussian(y_te, mu, sigma).mean()

## Visualize the data & results

_On real diabetes data, do 80% / 90% / 95% split-conformal intervals actually cover at their promised rates?_

We reuse the same split-conformal recipe, but now sweep three target coverages and watch what each one achieves. We do it in two steps: (1) refit and grab calibration residuals, (2) loop over the three targets, reporting achieved PICP and MPIW for each.

### Step 1 — Refit and collect calibration residuals

Same three-way split and same `LinearRegression` as before, on 442 real diabetes patients (the target is disease progression). We fit the model, then take the absolute residuals on the calibration set — these are the raw material every interval width is built from.

In [ ]:
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

X, y = load_diabetes(return_X_y=True)        # 442 real patients; target = progression
X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.4, random_state=0)
X_cal, X_te, y_cal, y_te = train_test_split(X_tmp, y_tmp, test_size=0.5, random_state=0)

model = LinearRegression().fit(X_tr, y_tr)
res = np.abs(y_cal - model.predict(X_cal))   # calibration residuals
pred = model.predict(X_te)                    # point predictions on the test set
n = len(res)

### Step 2 — Sweep three target coverages

For each promised coverage (0.80, 0.90, 0.95) we recompute the conformal half-width, build the interval, and measure what coverage it **actually** achieves on the untouched test set, alongside its width.

Notice the pattern: achieving a higher promised coverage requires a wider interval, and the achieved PICP tracks the target — the guarantee holding up on real data.

In [ ]:
for target in [0.80, 0.90, 0.95]:            # the three coverage levels
    alpha = 1 - target
    k = min(int(np.ceil((n + 1) * (1 - alpha))), n)
    qhat = np.sort(res)[k - 1]               # conformal half-width for this target

    lo = pred - qhat                          # interval lower bound
    hi = pred + qhat                          # interval upper bound

    inside = (y_te >= lo) & (y_te <= hi)
    picp = np.mean(inside)                    # achieved coverage
    mpiw = np.mean(hi - lo)                   # average width

    print(target, "PICP", round(float(picp), 3), "MPIW", round(float(mpiw), 1))
# 0.80 PICP 0.888 MPIW 154.2
# 0.90 PICP 0.921 MPIW 197.1
# 0.95 PICP 0.933 MPIW 226.3

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Team A reports "90% intervals, MPIW 6." Team B reports "90% intervals, MPIW 20." Which is better?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Refuse to compare on WIDTH alone — ask both teams for PICP on a held-out test set. — _Width is only meaningful once you know each model actually covers at the promised rate; a tight interval that misses is worthless._
- Suppose Team A's PICP is 0.72 and Team B's is 0.90. — _Team A's narrow intervals are narrow because they are too small — they cover only 72%, far below the 90% promise (over-confident)._
- Pick the one that COVERS first, then prefer narrower among those that do. — _Calibration (coverage matching the promise) is the gate; width is the tie-breaker._

**Answer:** You cannot say from width alone. Once you add coverage, Team B wins: it is calibrated (PICP 0.90) while Team A is over-confident (PICP 0.72) and only looks good because its intervals are too narrow. The rule: compare width only among intervals that cover at the target rate. Always quote PICP and MPIW together.

</details>

**Problem 2.** You have a fitted regression model and want 90% prediction intervals with a guarantee that doesn't assume Gaussian errors. Outline split conformal and what you'd check.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Hold out a CALIBRATION split separate from train and test. — _The coverage guarantee needs calibration residuals to be exchangeable with test residuals, so calibration data must be unseen by the fitted model._
- Compute absolute residuals $s_i = |y_i - \hat y_i|$ on the calibration split. — _These capture the model's typical error magnitude with no distributional assumption._
- Take $\hat q$ at rank $\lceil (n+1)(0.9)\rceil$ of the sorted residuals; set each interval to $\hat y \pm \hat q$. — _The $(n+1)$ correction makes the finite-sample $\ge 0.9$ coverage exact._
- Measure PICP and MPIW on the untouched TEST set, and also PICP per segment. — _Confirms achieved coverage ≈ 0.90 overall and exposes segments where conditional coverage is weak._

**Answer:** Split-conformal regression: fit on train, take absolute residuals on a fresh calibration split, set $\hat q$ to their $\lceil (n+1)(1-\alpha)\rceil$-th smallest value, and output $\hat y \pm \hat q$. Verify on the held-out test set that PICP ≈ 0.90 and report MPIW in the target's units. Also check coverage per segment — exchangeability buys marginal coverage, not automatic conditional coverage. In code, MAPIE's MapieRegressor automates this.

</details>

**Problem 3.** A weather team predicts a full probability distribution for tomorrow's temperature, not just an interval. How do you score it, and how is that different from pinball loss?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Use CRPS (Continuous Ranked Probability Score) — it scores the whole predicted distribution against the single observed temperature. — _CRPS measures the squared area between the predicted cumulative curve and the perfect step at the truth, rewarding sharp, well-placed distributions._
- Note CRPS is in the same units as temperature and reduces to MAE (Mean Absolute Error) for a point forecast. — _That makes it directly comparable to a deterministic baseline._
- Contrast with pinball loss, which scores ONE quantile at a time. — _Averaging pinball loss over many quantile levels approximates CRPS, but a single pinball value only judges one slice of the distribution._

**Answer:** Score the full forecast with CRPS: $\int (F(t)-\mathbb{1}[t\ge y])^2\,dt$, the squared gap between the predicted cumulative curve $F$ and the step that jumps at the truth $y$. It rewards distributions that are both sharp and correctly centered, lives in temperature units, and collapses to MAE for a point forecast. Pinball loss instead scores a single quantile $\tau$; averaging pinball over a grid of quantiles approximates CRPS. Use properscoring.crps_ensemble for CRPS.

</details>